In [2]:
import asyncio

# Why do we need asyncio?

In a program, there are broadly two kinds of time-consuming work. Some operations keep the CPU busy doing computation, while others spend most of their time waiting for something external to happen. Reading from a network, waiting for a server response, waiting for a database query, reading from a socket, and waiting for a timer are all examples of the second category.

Asyncio pays off when a program handles many operations that each spend most of their time waiting on I/O. Typical examples in system design are web servers and API gateways (FastAPI, aiohttp), services that fan out to several downstream services per request, web crawlers and scrapers, chat and notification servers holding thousands of open WebSocket connections, and message consumers that read from a queue and call other services. A single asyncio process can keep thousands of such operations in flight because a paused coroutine costs only a small amount of memory, whereas a thread per connection costs a full thread stack and operating-system scheduling overhead.
 
Asyncio does not help with CPU-bound work such as image processing, heavy parsing, or numerical computation. Because the event loop runs one job at a time on one thread, a long computation blocks every other task. The same is true of any blocking library call, such as `time.sleep`, the `requests` library, or a synchronous database driver. The following example shows the effect on a heartbeat task that should print every half second.
 

| Workload | Good fit for asyncio? | Reason |
|---|---|---|
| Many concurrent network calls or connections | Yes | Waits overlap on one thread with low memory cost per operation |
| A few slow blocking calls in an otherwise sync program | Often simpler with threads | Adding asyncio means async-compatible libraries throughout |
| CPU-heavy computation | No | One thread; use processes or a separate service |
| Mixed: async server that occasionally does blocking work | Yes, with `to_thread` or executors | Keeps the event loop responsive |

In [ ]:
def download_file():
    print("Starting download")

    # Imagine this takes 5 seconds because we are
    # waiting for a remote server to respond.
    time.sleep(5)

    print("Download complete")


def send_email():
    print("Sending email")

    # Imagine this also spends most of its time
    # waiting for a mail server.
    time.sleep(3)

    print("Email sent")


download_file()
send_email()

Instead of doing this:

    Download ────────────────┐
                             └── complete
    
                             Email ────────────┐
                                              └── complete

we would like the program to behave more like:

    Download ────────────────────────────────
              │
              │ waiting...
              │
              └───────────────┐
                              │
    Email ───────────────────┼───────────────
                             │
                             └── complete

In [ ]:
def download_file():
    print("Starting download")

    # Imagine this takes 5 seconds because we are
    # waiting for a remote server to respond.
    time.sleep(5)

    print("Download complete")


def send_email():
    print("Sending email")

    # Imagine this also spends most of its time
    # waiting for a mail server.
    time.sleep(3)

    print("Email sent")


download_file()
send_email()

In [4]:
async def tea():
    print("tea: start")
    await asyncio.sleep(2)
    print("tea: done")

async def toast():
    print("toast: start")
    await asyncio.sleep(1)
    print("toast: done")

async def main():
    t1 = asyncio.create_task(tea())
    t2 = asyncio.create_task(toast())
    await t1
    await t2

# asyncio.run(main())
await main() # jupyter notebook already has a event loop

tea: start
toast: start
toast: done
tea: done


# Voacabulary around asyncio 
    Coroutine : a piece of asynchronous work.
    Async/await: 
    Awaitable : something you are allowed to await.
    Task : a coroutine that has been handed to the event loop to be executed.
    Future : an object representing a result that will become available later.
    Event loop : the mechanism that keeps track of asynchronous work and decides what can run next.

## coroutine
A function whose execution can be paused and later resumed. In Python, we create one using `async def`.


A function defined with `async def` is a coroutine function. Calling it does not run its body. Instead, the call returns a coroutine object, which represents the function's body in a paused state, ready to start. The body only runs when something drives it, and the normal way to drive it from inside other async code is the `await` keyword.

What makes a coroutine different from a regular function is that its execution can be suspended partway through (at an `await`) and resumed later, with all of its local variables intact. That ability to pause is what lets the event loop run other work while one coroutine waits.

In [24]:
import asyncio

async def greet(name):
    print(f"Hello, {name}")
    return f"greeted {name}"

async def main():
    coro = greet("Ramesh")          # creates a coroutine object; nothing printed yet
    print("Created:", coro)
    result = await coro           # now the body runs
    print("Result:", result)

# asyncio.run(main())
await main()

Created: <coroutine object greet at 0x10738c930>
Hello, Ramesh
Result: greeted Ramesh


In [6]:
import asyncio


async def make_tea():
    print("Starting tea")

    await asyncio.sleep(2)

    print("Tea is ready")
    return "tea"

In [7]:
result = make_tea()

print(result)

<coroutine object make_tea at 0x1070c7640>


Calling a coroutine function does not mean that the coroutine has actually executed. `make_tea()` produces a coroutine object. 

Normal function executes immmediately but a corouting creates a coroutine object. 
Coroutine is asynchronous work that can be suspended and resumed. 

In [10]:
await make_tea()  # coroutines can be awaited. couroutine is awaitable. 

Starting tea
Tea is ready


'tea'

In [ ]:
Awaiting a Coroutine: Sequential Execution

await some_coroutine() behaves much like calling a regular function: the current coroutine starts the awaited one, waits for it to finish, and receives its return value. The difference is that while the awaited coroutine is waiting on I/O, the whole chain is paused and the event loop is free to run other jobs.

That last sentence contains a subtlety. The event loop is free to run other jobs, but if there are no other jobs, nothing else happens. Awaiting coroutines one after another produces sequential code. The following program makes three simulated network calls this way, then does the same work with tasks, which we introduce properly in the next section.

In [27]:
import asyncio, time

async def fetch(name, delay):
    print(f"{name}: start")
    await asyncio.sleep(delay)     # simulates waiting on the network
    print(f"{name}: done")
    return name

async def sequential():
    start = time.perf_counter()
    a = await fetch("A", 1)
    b = await fetch("B", 1)
    c = await fetch("C", 1)
    print(f"sequential -> {[a, b, c]} in {time.perf_counter() - start:.1f}s")

async def concurrent():
    start = time.perf_counter()
    ta = asyncio.create_task(fetch("A", 1))
    tb = asyncio.create_task(fetch("B", 1))
    tc = asyncio.create_task(fetch("C", 1))
    a = await ta
    b = await tb
    c = await tc
    print(f"concurrent -> {[a, b, c]} in {time.perf_counter() - start:.1f}s")

async def main():
    await sequential()
    print("-" * 30)
    await concurrent()

# asyncio.run(main())
await main()

A: start
A: done
B: start
B: done
C: start
C: done
sequential -> ['A', 'B', 'C'] in 3.0s
------------------------------
A: start
B: start
C: start
A: done
B: done
C: done
concurrent -> ['A', 'B', 'C'] in 1.0s


In [ ]:
## Awaitable
This represents asynchronous work/result that can be waited for using `await`.

Anything that can be awaited is awaitable. There are three types of awaitables: Coroutine, Task and Futures. 

In [ ]:
# this is a coroutine
async def make_tea():
    await asyncio.sleep(2)
    return "tea"

coro = make_tea() # coroutine object
await coro # awaiting a coroutine

## Task

Tldr ; A coroutine that has been scheduled for execution by the event loop.

A task is an object that wraps a coroutine and registers it with the event loop so the loop will run it on its own schedule. `asyncio.create_task(coro)` creates the task and places it in the event loop's ready queue immediately. From that moment the task runs independently of the code that created it: the event loop will start it at the next opportunity, whether or not anyone ever awaits it.

A task also stores the eventual outcome of its coroutine: the return value, or the exception it raised, or the fact that it was cancelled. You can inspect this with `task.done()`, `task.result()`, and `task.exception()`, and you can stop it with `task.cancel()`.

In [ ]:
async def make_tea():
    print("Starting tea")
    await asyncio.sleep(2)
    print("Tea is ready")


coro = make_tea()

In [ ]:
task = asyncio.create_task(make_tea())

In [ ]:
coro = make_tea() #creates a coroutine object.
task = asyncio.create_task(make_tea()) #creates a Task that schedules that coroutine to run

In [ ]:
# we can wait a task

result = await task

In [11]:
async def tea():
    print("Tea started")
    await asyncio.sleep(2)
    print("Tea finished")


async def toast():
    print("Toast started")
    await asyncio.sleep(1)
    print("Toast finished")

task1 = asyncio.create_task(tea()) # takes a coroutine and schedules it
task2 = asyncio.create_task(toast())

await task1
await task2

Tea started
Toast started
Toast finished
Tea finished


In [12]:
## writing the above code without creating task. Directly awaiting the coroutine
async def tea():
    print("Tea started")
    await asyncio.sleep(2)
    print("Tea finished")


async def toast():
    print("Toast started")
    await asyncio.sleep(1)
    print("Toast finished")

# task1 = asyncio.create_task(tea())
# task2 = asyncio.create_task(toast())

await tea()
await toast()

Tea started
Tea finished
Toast started
Toast finished


In [ ]:
## Coroutine vs Task
 - If both task and coroutine can be awaited then what is the difference between the two?

In [ ]:
import asyncio


async def tea():
    print("tea: start")
    await asyncio.sleep(2)
    print("tea: done")


async def toast():
    print("toast: start")
    await asyncio.sleep(1)
    print("toast: done")


async def main():
    await tea()
    await toast()


await main()

For the above code :

    main
     |
     | await tea()
     v
    tea starts
     |
     | await sleep(2)
     v
    tea is suspended
     |
     | ... no other application task exists ...
     |
    tea resumes
     |
     v
    tea finishes
     |
     v
    main resumes
     |
     | await toast()
     v
    toast starts

In [ ]:
import asyncio


async def tea():
    print("tea: start")
    await asyncio.sleep(2)
    print("tea: done")


async def toast():
    print("toast: start")
    await asyncio.sleep(1)
    print("toast: done")


async def main():
    t1 = asyncio.create_task(tea()) # scheduled on the event loop
    t2 = asyncio.create_task(toast()) # scheduled on the event loop

    await t1
    await t2


await main()

                  Event Loop
                      |
              +-------+-------+
              |               |
              v               v
            Task 1          Task 2
              |               |
             tea()           toast()

In [14]:
import asyncio


async def child():
    print("child: start")
    await asyncio.sleep(1)
    print("child: done")


async def other():
    print("other: start")
    await asyncio.sleep(0)
    print("other: done")


async def main():
    # `other()` is scheduled independently.
    other_task = asyncio.create_task(other())

    # `child()` is only a coroutine. It has NOT been scheduled as an independent Task.
    await child()

    print("main: done")


await main()

child: start
other: start
other: done
child: done
main: done


In [20]:
async def coro_fn_a(i):
    print("I am coro_a(). Hi! -- ", i)
    await asyncio.sleep(1)
    print(f"I was {i}  and I am done now!!!")


async def coro_fn_b():
    print("I am coro_b().")


async def main():
    task_b = asyncio.create_task(coro_fn_b())

    for i in range(5):
        coro_a = coro_fn_a(i)
        await coro_a

    await task_b

await main()

I am coro_a(). Hi! --  0
I am coro_b().
I was 0  and I am done now!!!
I am coro_a(). Hi! --  1
I was 1  and I am done now!!!
I am coro_a(). Hi! --  2
I was 2  and I am done now!!!
I am coro_a(). Hi! --  3
I was 3  and I am done now!!!
I am coro_a(). Hi! --  4
I was 4  and I am done now!!!


In [21]:
import asyncio
import time


async def task1():
    print("Task 1 started")
    await asyncio.sleep(2)
    print("Task 1 finished")


async def task2():
    print("Task 2 started")
    await asyncio.sleep(2)
    print("Task 2 finished")


async def sequential():
    start = time.perf_counter()

    # Each coroutine is awaited before the next one is started.
    await task1()
    await task2()

    elapsed = time.perf_counter() - start
    print(f"Sequential time: {elapsed:.2f} seconds")


async def concurrent():
    start = time.perf_counter()

    # Both coroutines are scheduled immediately as Tasks.
    t1 = asyncio.create_task(task1())
    t2 = asyncio.create_task(task2())

    # Now wait for their completion.
    await t1
    await t2

    elapsed = time.perf_counter() - start
    print(f"Concurrent time: {elapsed:.2f} seconds")


async def main():
    print("----- Sequential -----")
    await sequential()

    print("\n----- Concurrent -----")
    await concurrent()


await main()

----- Sequential -----
Task 1 started
Task 1 finished
Task 2 started
Task 2 finished
Sequential time: 4.00 seconds

----- Concurrent -----
Task 1 started
Task 2 started
Task 1 finished
Task 2 finished
Concurrent time: 2.01 seconds


| | Coroutine object | Task |
|---|---|---|
| Created by | Calling an `async def` function | `asyncio.create_task(coro)`, `TaskGroup.create_task(coro)`, or internally by `asyncio.gather` |
| Starts running | Only when awaited (or otherwise driven) | Automatically, at the event loop's next opportunity |
| Scheduled by the event loop on its own | No | Yes |
| Can run concurrently with its creator | No | Yes |
| Holds a result you can inspect later | No | Yes (`done()`, `result()`, `exception()`) |
| Can be cancelled | Not directly | Yes (`cancel()`) |
| Can be awaited more than once | No | Yes (later awaits return the stored result) |

This separation between starting work and collecting its result is the whole reason tasks exist. With a bare coroutine, starting and waiting are the same step (await coro does both). With a task, you start the work at one point in your code and wait for it at a later point, and anything you do in between runs while the task makes progress.

| | `await coroutine` | `await task` |
|---|---|---|
| What happens first | The coroutine's body starts running immediately, inside the current coroutine | Nothing starts; the task was already scheduled when it was created |
| Does the act of awaiting hand control to the event loop? | No. Control passes directly into the coroutine's body, like a function call | Yes, if the task is not finished yet |
| Can other work overlap with it? | Only work that was already scheduled as tasks, and only while the coroutine is paused on real I/O | Yes, the task and other tasks run while you wait |
| Typical meaning | "Do this step now, then continue" | "I started this earlier; now I need its result" |

#### Experiment 1: awaiting coroutines that never pause

In [29]:
import asyncio
 
async def worker_a():
    print("  worker_a ran")
 
async def worker_b():
    print("  worker_b ran (the background task)")
 
async def await_coroutines():
    print("Awaiting coroutines:")
    task_b = asyncio.create_task(worker_b())   # scheduled, not yet running
    for _ in range(3):
        await worker_a()                       # runs inline, no hand-off to the loop
    await task_b
 
async def await_tasks():
    print("Awaiting tasks:")
    task_b = asyncio.create_task(worker_b())
    for _ in range(3):
        await asyncio.create_task(worker_a())  # hands control to the event loop
    await task_b
 
async def main():
    await await_coroutines()
    await await_tasks()
 
# asyncio.run(main())
await main()

Awaiting coroutines:
  worker_a ran
  worker_a ran
  worker_a ran
  worker_b ran (the background task)
Awaiting tasks:
  worker_b ran (the background task)
  worker_a ran
  worker_a ran
  worker_a ran


In the first half, `task_b` was ready to run from the start, yet it ran last. Each `await worker_a()` executed `worker_a`'s body directly and returned, and at no point did control go back to the event loop, so the loop never had a chance to start `task_b`. The loop only regained control at `await task_b`, because `task_b` was unfinished.
 
In the second half, the first `await asyncio.create_task(worker_a())` suspended `main` and returned control to the loop. The loop's ready queue at that moment held `task_b` (queued first) and then the new `worker_a` task, so `task_b` ran first.
 

#### Experiment 2: awaiting a coroutine that does pause

 
The first experiment might suggest that awaiting a coroutine can never let other work run. That conclusion is wrong, and this experiment shows why.

In [30]:

import asyncio
 
async def slow_step():
    print("  slow_step: about to wait on a timer")
    await asyncio.sleep(0.1)       # a real suspension point
    print("  slow_step: resumed")
 
async def background():
    print("  background task got a turn")
 
async def main():
    bg = asyncio.create_task(background())
    await slow_step()              # awaiting a coroutine...
    await bg
 
# asyncio.run(main())
await main()


  slow_step: about to wait on a timer
  background task got a turn
  slow_step: resumed


`main` awaited a coroutine, and the background task still ran in the middle of it. The `await slow_step()` did not pause anything by itself, but inside `slow_step` execution reached `asyncio.sleep(0.1)`, which must wait. At that point the entire chain (`slow_step` and `main` with it) was suspended, and the event loop ran the background task. This is the normal situation in real programs: the coroutines you await are usually doing I/O, so they do pause, and other tasks do run while they wait.

In [ ]:
#### Experiment 3: awaiting a task that has already finished
 
The second half of the claim, "awaiting a task yields control", also has an exception.

In [31]:


import asyncio
 
async def quick():
    return 42
 
async def other():
    print("  other task ran")
 
async def main():
    t = asyncio.create_task(quick())
    await asyncio.sleep(0)          # let t run to completion
    print("t done?", t.done())
 
    o = asyncio.create_task(other())
    print("awaiting a task that is already finished...")
    value = await t                 # no suspension: result is returned immediately
    print("got", value, "and 'other' has not run yet")
    await o
 
# asyncio.run(main())
await main()

t done? True
awaiting a task that is already finished...
got 42 and 'other' has not run yet
  other task ran


In [ ]:
## Future
A Future is an object representing a result that is not available yet (placeholder for a result that will be available later)

## Event loop

In [ ]:
The event loop is responsible for repeatedly checking asynchronous work and determining what can make progress.

it rapidly switches between pieces of asynchronous work whenever they reach points where they cannot currently make progress.

In [ ]:
async def tea():
    await asyncio.sleep(2)
    return "tea"


async def main():
    task = asyncio.create_task(tea())

    result = await task

    print(result)

#### When to Use Which
 
With the mechanism clear, the practical guidance follows directly.
 
**Await a coroutine directly** when the next line of your code needs the result before it can continue. This is the most common form of `await` in real code by a wide margin. Reading a request body, then validating it, then writing to the database, then returning a response is a sequence of steps, and each step is `await some_coroutine()`. Other requests handled by the same server still run concurrently, because each request is handled in its own task created by the web framework.
 
**Create a task** when you want work to proceed while your coroutine does something else. The typical cases are:
 
- several independent I/O operations whose results you need together (the dashboard example);
- background work that should keep running while the main flow continues, such as a periodic heartbeat or a cache refresher;
- work that you may need to cancel or time out independently.
**Await the task later** when you need its result or need to know it has finished. Creating tasks and awaiting them immediately (`await asyncio.create_task(coro)`) gains nothing over `await coro`, because nothing else is started in between.


In modern Python you usually do not call `create_task` and then await each task by hand. Two higher-level tools do it for you:
 
- `asyncio.gather(*coros)` wraps each coroutine in a task, runs them concurrently, and returns their results as a list in the same order.
- `asyncio.TaskGroup` (Python 3.11+) is a context manager. Tasks created inside the `async with` block are all awaited when the block exits. If one task fails, the group cancels the others and raises the errors together. This is called **structured concurrency**: no task can outlive the block that created it, which prevents forgotten tasks.


| Situation | What to write |
|---|---|
| Next step needs this result | `result = await do_step()` |
| Several independent calls, need all results | `async with asyncio.TaskGroup()` or `await asyncio.gather(...)` |
| Start something, do other work, collect it later | `t = asyncio.create_task(...)`, then `await t` |
| Long-lived background job | A task, kept in a variable or a set, cancelled and awaited on shutdown |
| Need a timeout on one operation | `async with asyncio.timeout(seconds):` around the await (Python 3.11+) |

#### Under the Hood: How a Pause Travels Up the Chain
 
This section explains the mechanism behind the rule above. It is optional for day-to-day use, but it removes any mystery from the behaviour we observed.
 
Coroutines are built on the same machinery as Python generators. A generator pauses at `yield` and hands a value to whoever is iterating it. A coroutine chain works the same way: `await x` delegates to `x`, and if somewhere at the bottom of the chain an object executes a `yield`, that yielded value travels up through every coroutine in the chain until it reaches whatever is driving the outermost coroutine. In a real program the driver is a task, which is itself driven by the event loop. When nothing in the chain yields, `await` simply runs the inner code to completion and returns its value, which is why awaiting a coroutine behaves like a function call.
 
The following program removes the event loop entirely and drives a three-level coroutine chain by hand, using `send()`, the same method the event loop uses internally.
 
```python
# Drive a chain of coroutines by hand, with no event loop, to see what `await` does.
 
class Suspend:
    """The smallest possible awaitable that actually pauses."""
    def __await__(self):
        # `yield` hands a value all the way up to whoever is driving the coroutine.
        reply = yield "suspend: wake me when the data arrives"
        return reply
 
async def leaf():
    print("    leaf: about to wait for data")
    data = await Suspend()          # the only real pause point in the whole chain
    print(f"    leaf: resumed with {data!r}")
    return data.upper()
 
async def middle():
    print("  middle: calling leaf() with await")
    result = await leaf()           # no pause of its own, just delegates
    print(f"  middle: leaf returned {result!r}")
    return result + "!"
 
async def top():
    print("top: calling middle() with await")
    return await middle()
 
coro = top()
 
# Step 1: start the coroutine. It runs until something inside it yields.
message = coro.send(None)
print(f"DRIVER got: {message!r}")
 
# Step 2: resume it, passing in the "data" the leaf was waiting for.
try:
    coro.send("hello")
except StopIteration as done:
    print(f"DRIVER: coroutine finished with {done.value!r}")
```
 
Output:
 
```text
top: calling middle() with await
  middle: calling leaf() with await
    leaf: about to wait for data
DRIVER got: 'suspend: wake me when the data arrives'
    leaf: resumed with 'hello'
  middle: leaf returned 'HELLO'
DRIVER: coroutine finished with 'HELLO!'
```
 
A single `send(None)` ran straight through `top`, `middle`, and into `leaf`. The two coroutine awaits (`await middle()` and `await leaf()`) caused no pause. Execution stopped only at the `yield` inside `Suspend`, and the yielded message arrived directly at the driver, passing through all three coroutines. When the driver called `send("hello")`, the value travelled back down to the `yield`, and the chain finished normally, with each level receiving its return value from the level below.
 
This is exactly what the event loop does. In real asyncio, the object at the bottom of the chain is a **future**: a placeholder for a result that is not available yet. `asyncio.sleep`, network reads, and unfinished tasks all eventually await a future. When a future is not done, it yields itself up the chain to the task driving the chain. The task then attaches a callback to the future saying "resume me when you are done" and returns control to the event loop. When the timer fires or the data arrives, the future is marked done, the callback puts the task back in the ready queue, and the event loop calls `send()` on the chain again. A task, in other words, is the driver that the event loop uses for one coroutine chain.

# Task Groups


```python
import asyncio, time
 
async def fetch_user(user_id):
    await asyncio.sleep(1.0)                 # pretend: a database query
    return {"id": user_id, "name": f"user{user_id}"}
 
async def fetch_orders(user_id):
    await asyncio.sleep(1.5)                 # pretend: a call to the orders service
    return [f"order-{user_id}-1", f"order-{user_id}-2"]
 
async def fetch_recommendations(user_id):
    await asyncio.sleep(1.2)                 # pretend: a call to the ML service
    return ["book", "lamp"]
 
async def build_dashboard(user_id):
    start = time.perf_counter()
    # The three calls are independent, so run them concurrently in a TaskGroup.
    async with asyncio.TaskGroup() as tg:
        user_task = tg.create_task(fetch_user(user_id))
        orders_task = tg.create_task(fetch_orders(user_id))
        recs_task = tg.create_task(fetch_recommendations(user_id))
    # Leaving the `async with` block waits for every task in the group.
    elapsed = time.perf_counter() - start
    return {
        "user": user_task.result(),
        "orders": orders_task.result(),
        "recommendations": recs_task.result(),
        "took": f"{elapsed:.1f}s",
    }
 
async def main():
    print(await build_dashboard(7))
 
asyncio.run(main())
```
 
Output:
 
```text
{'user': {'id': 7, 'name': 'user7'}, 'orders': ['order-7-1', 'order-7-2'], 'recommendations': ['book', 'lamp'], 'took': '1.5s'}
```
 
The response took 1.5 seconds, the duration of the slowest call, instead of the 3.7-second sum. Notice how the two forms of `await` divide the work: `main` awaits the coroutine `build_dashboard(7)` directly because it needs the result before printing, while `build_dashboard` uses tasks because its three calls are independent. After the block exits, every task is guaranteed to be done, so calling `.result()` is safe.
 

# Practice Questions
 
**Question 1.** Predict the output of this program, and state how long it takes to run.
 
```python
import asyncio
 
async def job(name, delay):
    await asyncio.sleep(delay)
    print(name)
 
async def main():
    t = asyncio.create_task(job("task", 1))
    await job("coroutine", 2)
    print("main continues")
    await t
 
asyncio.run(main())
```
**Solution 1.** The output is:
 
```text
task
coroutine
main continues
```

The task is scheduled at once. `await job("coroutine", 2)` runs the coroutine inline until it reaches `asyncio.sleep(2)`, where the chain suspends and the event loop runs the task. The task's one-second sleep finishes first and prints `task`. After two seconds the coroutine prints `coroutine`, `main` continues, and `await t` returns immediately because the task has already finished. The total is about two seconds, since the two sleeps overlapped.


---
**Question 2.** A teammate "optimises" a request handler by rewriting every `await do_step()` as `await asyncio.create_task(do_step())`. The handler performs five dependent steps in order. Will the handler get faster? Explain what changes and what does not.

**Solution 2.** The handler will not get faster, and it becomes slightly slower. Each step is awaited immediately after its task is created, so no two steps are ever in flight at the same time; the steps are dependent anyway, so they cannot overlap. The change only adds the overhead of creating a task and passing through the event loop for each step. Other requests were already able to run during each step's I/O, because each request lives in its own task, so fairness does not improve either.

---

**Question 3.** An async web server has an endpoint that generates a report by looping over two million rows in pure Python and computing statistics, which takes about eight seconds of CPU time. Someone wraps the computation in an `async def` function and starts it with `asyncio.create_task` so that "it runs in the background". What happens to the other requests being served by that process, and what would you do instead?


**Solution 3.** The computation is CPU-bound and contains no pause points, so once the task starts it holds the event loop for all eight seconds. Every other request on that process freezes for that time, including health checks, which may cause a load balancer to mark the instance as unhealthy. Making it a task only changes when the computation starts; it still runs on the event loop's single thread. The fix is to move the work off the event loop. Running it in a separate process with `await loop.run_in_executor(process_pool, compute_report, rows)` keeps the loop free and uses another CPU core. For long jobs, a common system-design choice is to enqueue the work to a background worker service and return a job ID that the client polls, so web servers never perform heavy computation at all. A thread (`asyncio.to_thread`) is a weaker option here: pure-Python code in a thread still competes with the event loop for Python's global interpreter lock (GIL), so the server stays responsive but slower.


---
**Question 4 (scenario).** You are designing a notification service. For each incoming event, it must look up the user's preferences (database, about 20 ms), then send an email (about 300 ms), a push notification (about 150 ms), and an SMS (about 400 ms). The three sends depend on the preferences but not on each other. Sketch the structure of the async handler, stating which calls you await directly and which you run as tasks, and estimate the per-event latency. How would you handle one of the three channels failing?


**Solution 4.** The preferences lookup is a dependency of everything else, so it is awaited directly: `prefs = await load_preferences(user_id)`. The three sends are independent, so they run concurrently in a `TaskGroup` (or `gather`). Latency is roughly 20 ms plus the slowest send, 400 ms, giving about 420 ms, compared with about 870 ms sequentially. Failure handling is a design decision. With a `TaskGroup`, one failure cancels the other sends, which is usually undesirable for notifications, since a failed SMS should not cancel a successful email. A better approach is `asyncio.gather(..., return_exceptions=True)`, or wrapping each send in its own `try/except` inside the task, so that each channel's failure is recorded and retried independently (for example by publishing a retry message to a queue), while the other channels complete. Each send should also have its own timeout with `asyncio.timeout` so that a hung provider cannot hold the handler indefinitely.

---

**Question 5.** In the program below, `worker` never prints. Explain why, and fix it in two different ways.
 
```python
import asyncio
 
async def worker():
    await asyncio.sleep(0.1)
    print("worker finished")
 
async def main():
    asyncio.create_task(worker())
    print("main finished")
 
asyncio.run(main())
```

**Solution 5.** `main` creates the task and returns immediately. `asyncio.run` then considers the program finished and cancels any remaining tasks while shutting down the loop, so the worker is cancelled during its sleep and never prints. The task was also unreferenced, so it could have been garbage-collected. Two fixes:
 
```python
# Fix A: keep a reference and await it
async def main():
    t = asyncio.create_task(worker())
    print("main finished")
    await t
 
# Fix B: structured concurrency with a TaskGroup
async def main():
    async with asyncio.TaskGroup() as tg:
        tg.create_task(worker())
        print("main finished")
```
 
In both versions, `main` does not complete until the worker has finished, so the output is `main finished` followed by `worker finished`.

---

 

 

 

 

 


In [ ]:
Other topics:
    - asyncio.to_thread
    - asyncio.to_process
    - aiohttp / httpx
    - other alternative to blocking libraries and async alternatives to them
    - 